# 🚀 LLM Evaluation und Prompt-Tuning

> Willkommen! In diesem Notebook lernst du:
> 1. Wie man ein LLM aufruft
> 2. Dass LLMs **nicht perfekt** sind
> 3. Wie man **Accuracy misst**
> 4. Wie du **selbst Prompts verbesserst** — und warum das mühsam ist
>
# Wie navigiere ich durch dieses Notebook?
> 1. Lies was steht
> 2. Drücke Shift-Enter um den Code auszuführen.
> 3. Experimentiere mit dem Prompts und den Daten.
>
# Häufige Fehler
> Die Kasten müssen von oben nach unten ausgeführt werden.
> Zuerst musst du einmal das ./start.sh laufen lassen.


## Worum geht's hier?

Dieses Notebook hat zwei Teile:

1. **Teil 1** — Wir rufen ein LLM auf, sehen wo es Fehler macht, und lernen 5 Strategien dagegen
2. **Teil 2** — Wir messen Qualität mit Metriken und tunen Prompts — erst manuell, dann mit Benchmarks

Am Ende weisst du: Wie gut ist mein Modell? Und wie mache ich es besser?

> **Tipp:** Jede Code-Zelle kannst du mit Shift+Enter ausführen. Die Ergebnisse erscheinen direkt darunter.


## 🛠️ Setup

Wir importieren die nötigen Bibliotheken und unsere Task-Bibliothek.
- **Tasks**: 20 kuratierte Aufgaben in 4 Schwierigkeitsstufen
- **Visualisierungen**: Interaktive Widgets zum Experimentieren
- **Actions**: Funktionen die LLMs aufrufen und Ergebnisse auswerten

In [1]:
import sys
sys.path.insert(0, ".")  # notebooks/ is the working dir
import dspy
from dspy_tasks.tasks import list_tasks, task_summary, get_task
from dspy_tasks.visualize import model_picker, display_score, display_insight, display_tier_header

## 🗺️ Die Reise durch alle Notebooks

Hier siehst du die fünf Stationen auf einen Blick. Wir starten mit **Evaluation** — also der Frage: *Wie gut ist mein Modell eigentlich?*

Jedes Notebook baut auf dem vorherigen auf. Am Ende hast du ein komplettes Bild davon, wie man KI-Systeme systematisch verbessert.


In [2]:
from dspy_tasks.visualize import diagram
diagram([
    {"label": "01 Evaluation", "detail": "LLM-Fehler + Metriken + Tuning", "icon": "📐", "color": "#0078d4"},
    {"label": "02 Optimierung", "detail": "Manuell vs. automatisch", "icon": "⚙️", "color": "#ca5010"},
    {"label": "03 Domain-Daten", "detail": "Dein Burggraben", "icon": "🏰", "color": "#ca5010"},
    {"label": "04 Agenten", "detail": "Tool-Nutzung", "icon": "🤖", "color": "#107c10"},
    {"label": "05 Gesamtbild", "detail": "Showdown + Quiz", "icon": "🎯", "color": "#107c10"},
], title="Dein Lernpfad")


## 🎯 Wähl dein Modell

Über LiteLLM kannst du über 100 verschiedene Modelle ansprechen. Die verfügbaren Modelle werden automatisch aus deiner `.env`-Konfiguration erkannt.

In [3]:
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

# Available models (add/remove based on your setup)
AVAILABLE_MODELS = get_available_models()

model_dropdown = model_picker(AVAILABLE_MODELS, default=get_default_model())
display(model_dropdown)

Dropdown(description='Model:', layout=Layout(width='400px'), options=('github_copilot/gpt-4o', 'github_copilot…

## 1. Wie gut ist das Modell wirklich?

Lass uns das Modell auf **Fragen mit bekannter Antwort** testen. Wir schauen genau hin: wo liegt es richtig — und wo **daneben**?


In [15]:
import dspy
from dspy_tasks.config import configure_dspy

configure_dspy(model=model_dropdown.value)

# Fragen mit BEKANNTER richtiger Antwort
test_questions = [
    ("Was ist die Hauptstadt von Australien?",  "Canberra"),
    ("Wie viele Planeten hat unser Sonnensystem?", "8"),
    ("Ist Glas eine Flüssigkeit?", "Nein"),
    ("Können Goldische nur 3 Sekunden erinnern?", "Nein"),
    ("Sieht man die Chinesische Mauer vom Weltraum?", "Nein"),
    ("Was ist schwerer: 1kg Stahl oder 1kg Federn?", "Gleich schwer"),
    ("Wie viel Prozent des Gehirns nutzen Menschen?", "100%"),
]

class QA(dspy.Signature):
    """Beantworte die Frage kurz und korrekt."""
    question = dspy.InputField(desc="Eine Wissensfrage")
    answer = dspy.OutputField(desc="Kurze, korrekte Antwort")

qa = dspy.Predict(QA)

print("Frage | Erwartete Antwort | Modell-Antwort | Match?")
print("=" * 80)

for question, expected in test_questions:
    result = qa(question=question)
    model_answer = result.answer.strip()
    exact = expected.lower() == model_answer.lower()
    contains = expected.lower() in model_answer.lower()
    
    if exact:
        icon = "✅ Exakt"
    elif contains:
        icon = "🟡 Enthalten"
    else:
        icon = "❓ Unklar"
    
    print(f"{icon}")
    print(f"   Frage:    {question}")
    print(f"   Erwartet: {expected}")
    print(f"   Modell:   {model_answer[:80]}")
    print()


Frage | Erwartete Antwort | Modell-Antwort | Match?
✅ Exakt
   Frage:    Was ist die Hauptstadt von Australien?
   Erwartet: Canberra
   Modell:   Canberra

✅ Exakt
   Frage:    Wie viele Planeten hat unser Sonnensystem?
   Erwartet: 8
   Modell:   8

🟡 Enthalten
   Frage:    Ist Glas eine Flüssigkeit?
   Erwartet: Nein
   Modell:   Nein, Glas ist ein amorpher Feststoff, keine Flüssigkeit.

🟡 Enthalten
   Frage:    Können Goldische nur 3 Sekunden erinnern?
   Erwartet: Nein
   Modell:   Nein, Goldfische können sich über Wochen bis Monate erinnern, nicht nur 3 Sekund

🟡 Enthalten
   Frage:    Sieht man die Chinesische Mauer vom Weltraum?
   Erwartet: Nein
   Modell:   Nein, mit bloßem Auge ist die Mauer weder vom Mond noch aus dem All klar sichtba

❓ Unklar
   Frage:    Was ist schwerer: 1kg Stahl oder 1kg Federn?
   Erwartet: Gleich schwer
   Modell:   Beide wiegen gleich viel, nämlich 1 kg.

❓ Unklar
   Frage:    Wie viel Prozent des Gehirns nutzen Menschen?
   Erwartet: 100%
   Model

## 💡 Siehst du das Problem?

Schau dir die Ergebnisse oben genau an. Das Modell antwortet zum Beispiel:

- Wir erwarten **"8"** → Modell sagt **"Acht."** — Ist das richtig? Ja! Aber ein einfacher String-Vergleich sagt: ❌
- Wir erwarten **"Gleich schwer"** → Modell sagt **"Beides wiegt gleich viel"** — Inhaltlich korrekt, aber komplett anderer Text!
- Wir erwarten **"Nein"** → Modell sagt **"Nein, Glas ist ein amorpher Feststoff..."** — Richtig, aber viel zu lang

**Das ist das fundamentale Problem:** LLMs geben jedes Mal einen **anderen String** zurück — selbst wenn die Antwort inhaltlich stimmt. Wie willst du da automatisch messen, ob die Antwort richtig ist?

Genau dafür brauchst du **Metriken** — clevere Funktionen die nicht nur `==` vergleichen, sondern verstehen, ob die *Bedeutung* stimmt.

Das ist das Thema der nächsten Sektion. 👇


## 2. Wie kann man LLM-Antworten automatisch bewerten?

Das Problem hast du gesehen: das Modell sagt "Acht" statt "8" — inhaltlich richtig, aber ein String-Vergleich scheitert. Hier sind 5 Strategien, wie man das löst:


## 🔧 Wie löst man das? — 5 Strategien

### 1. Antwort einschränken
Statt dem Modell freien Text zu erlauben, gibst du vor: **"Antworte nur mit Ja oder Nein"** oder **"Antworte mit einer Zahl"**. Dann reicht ein einfacher Vergleich.

### 2. Multiple Choice
Du gibst die möglichen Antworten vor: **A, B, C oder D**. Das Modell muss einen Buchstaben wählen — leicht zu prüfen.

### 3. Schlüsselwörter suchen
Statt exaktem Match prüfst du, ob bestimmte Schlüsselwörter in der Antwort vorkommen. "Canberra" in "Die Hauptstadt ist Canberra" → ✅

### 4. Semantischer Vergleich
Du nutzt Embeddings oder eine Ähnlichkeits-Metrik (z.B. Token-F1) um zu messen, wie *ähnlich* die Antwort der erwarteten ist — auch wenn die Worte anders sind.

### 5. Ein LLM als Richter (LLM-as-Judge)
Du lässt ein **zweites LLM** bewerten: *"Sagt diese Antwort inhaltlich das Gleiche wie die Referenz?"* Das klingt verrückt, funktioniert aber erstaunlich gut.


### Strategie 1 ausprobieren: Antworten einschränken

Wenn wir dem Modell sagen "antworte nur mit einem Wort", wird der Vergleich trivial:


In [16]:
# Lass uns Strategie 1 und 5 direkt ausprobieren!

# --- Strategie 1: Antwort einschränken ---
class StrictQA(dspy.Signature):
    """Beantworte die Frage. Antworte NUR mit einer Zahl, 'Ja', 'Nein', oder einem einzelnen Wort."""
    question = dspy.InputField(desc="Eine Wissensfrage")
    answer = dspy.OutputField(desc="Nur ein Wort oder eine Zahl, nichts sonst")

strict_qa = dspy.Predict(StrictQA)

print("━" * 50)
print("Strategie 1: Eingeschränkte Antworten")
print("━" * 50)

test_pairs = [
    ("Wie viele Planeten hat unser Sonnensystem?", "8"),
    ("Ist Glas eine Flüssigkeit?", "nein"),
    ("Was ist die Hauptstadt von Australien?", "canberra"),
]

for q, expected in test_pairs:
    result = strict_qa(question=q)
    answer = result.answer.strip().lower().rstrip('.')
    match = answer == expected.lower()
    icon = "✅" if match else "❌"
    print(f"  {icon} {q}")
    print(f"     Modell: '{result.answer.strip()}' | Erwartet: '{expected}' | Exakt: {match}")
    print()

print("👆 Viel einfacher zu vergleichen wenn die Antwort eingeschränkt ist!")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategie 1: Eingeschränkte Antworten
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ Wie viele Planeten hat unser Sonnensystem?
     Modell: '8' | Erwartet: '8' | Exakt: True

  ✅ Ist Glas eine Flüssigkeit?
     Modell: 'Nein' | Erwartet: 'nein' | Exakt: True

  ✅ Was ist die Hauptstadt von Australien?
     Modell: 'Canberra' | Erwartet: 'canberra' | Exakt: True

👆 Viel einfacher zu vergleichen wenn die Antwort eingeschränkt ist!


### Strategie 5 ausprobieren: LLM als Richter

Ein zweites LLM bewertet, ob die Antwort **inhaltlich** korrekt ist — auch wenn die Worte ganz anders sind:


In [17]:
# --- Strategie 5: LLM-as-Judge ---
class Judge(dspy.Signature):
    """Bewerte ob die gegebene Antwort inhaltlich korrekt ist. Antworte NUR mit 'korrekt' oder 'falsch'."""
    question = dspy.InputField(desc="Die ursprüngliche Frage")
    expected_answer = dspy.InputField(desc="Die Referenz-Antwort")
    model_answer = dspy.InputField(desc="Die zu bewertende Antwort")
    verdict = dspy.OutputField(desc="Nur 'korrekt' oder 'falsch'")

judge = dspy.Predict(Judge)

print("━" * 50)
print("Strategie 5: LLM als Richter")
print("━" * 50)

judge_cases = [
    ("Wie viele Planeten?", "8", "Acht."),
    ("Was ist schwerer: 1kg Stahl oder 1kg Federn?", "Gleich schwer", "Beides wiegt gleich viel."),
    ("Gehirn-Nutzung?", "100%", "Menschen nutzen nahezu alle Teile ihres Gehirns."),
    ("Hauptstadt Australien?", "Canberra", "Sydney"),
]

for q, expected, model_ans in judge_cases:
    result = judge(question=q, expected_answer=expected, model_answer=model_ans)
    verdict = result.verdict.strip().lower()
    icon = "✅" if "korrekt" in verdict else "❌"
    print(f"  {icon} Frage: {q}")
    print(f"     Erwartet: '{expected}' | Modell sagte: '{model_ans}'")
    print(f"     Richter-Urteil: {result.verdict.strip()}")
    print()

print("👆 Der LLM-Richter versteht, dass 'Acht' == '8' und 'Beides wiegt gleich' == 'Gleich schwer'!")
print("   Aber: er kann sich auch irren — und er kostet einen extra API-Aufruf pro Bewertung.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategie 5: LLM als Richter
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ Frage: Wie viele Planeten?
     Erwartet: '8' | Modell sagte: 'Acht.'
     Richter-Urteil: korrekt

  ✅ Frage: Was ist schwerer: 1kg Stahl oder 1kg Federn?
     Erwartet: 'Gleich schwer' | Modell sagte: 'Beides wiegt gleich viel.'
     Richter-Urteil: korrekt

  ✅ Frage: Gehirn-Nutzung?
     Erwartet: '100%' | Modell sagte: 'Menschen nutzen nahezu alle Teile ihres Gehirns.'
     Richter-Urteil: korrekt

  ❌ Frage: Hauptstadt Australien?
     Erwartet: 'Canberra' | Modell sagte: 'Sydney'
     Richter-Urteil: falsch

👆 Der LLM-Richter versteht, dass 'Acht' == '8' und 'Beides wiegt gleich' == 'Gleich schwer'!
   Aber: er kann sich auch irren — und er kostet einen extra API-Aufruf pro Bewertung.


### 📊 Jede Strategie hat Tradeoffs

| Strategie | Vorteile | Nachteile |
|-----------|----------|----------|
| Antwort einschränken | Einfach, schnell, deterministisch | Geht nur bei geschlossenen Fragen |
| Multiple Choice | Leicht zu prüfen | Muss Optionen vorgeben |
| Schlüsselwörter | Flexibler als exakt | Kann false positives haben |
| Semantischer Vergleich | Erkennt Paraphrasen | Braucht Embeddings/Berechnung |
| LLM-as-Judge | Versteht Bedeutung | Kostet extra, kann sich auch irren |

In der Praxis kombiniert man diese Strategien — je nach Aufgabe. Genau das machen die **Metrik-Funktionen**, die wir gleich kennenlernen.


## 3. Metriken in der Praxis

In der Praxis nutzt man verschiedene Metriken je nach Aufgabe — von simplem Exact Match bis hin zu gewichteten Composite-Scores.


In [18]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    {"title": "Klassische Software", "items": ["Unit Test", "assert x == y", "Pass/Fail"], "icon": "🔧", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Metrik-Funktion", "metric(gold, pred) → score", "0.0 bis 1.0"], "icon": "🧠", "color": "#0078d4"},
    title="Tests vs. Metriken"
)

### Die verschiedenen Metrik-Level

Metriken werden immer raffinierter. Hier siehst du drei Level — von einfach bis komplex:


In [9]:
from dspy_tasks.calculations import token_f1

print("━" * 50)
print("📏 Level 1: Exact Match")
print("  Der einfachste Check: stimmt die Antwort exakt?")
print(f"  'positive' == 'positive' → 1.0 ✅")
print(f"  'positive' == 'negative' → 0.0 ❌")
print()
print("━" * 50)
print("📏 Level 2: Token F1 (Partial Credit)")
print("  Was wenn die Antwort TEILWEISE stimmt?")
print("  F1 balanciert Precision (wie viele Treffer waren korrekt?)")
print("  und Recall (wie viele wurden gefunden?)")
print()

cases = [
    (["apple", "banana", "cherry"], ["apple"],
     "Vorsichtig: nur 1 von 3 gefunden, aber der war richtig"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry", "grape", "melon"],
     "Übereifrig: alles gefunden, aber 2 Falsche dabei"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry"],
     "Perfekt: genau richtig"),
    (["apple", "banana"], ["cherry", "grape"],
     "Komplett daneben: nichts stimmt"),
]
for gold, pred, desc in cases:
    f1 = token_f1(gold, pred)
    icon = "✅" if f1 == 1.0 else "🟡" if f1 > 0 else "❌"
    print(f"  {icon} {desc}")
    print(f"     Gold: {gold} | Pred: {pred} → F1 = {f1:.2f}")

print()
print("━" * 50)
print("📏 Level 3: Composite (Ticket Routing)")
print("  Mehrere Kriterien mit Gewichtung:")
print("  Priorität korrekt (40%) + Kategorie (35%) + Team (25%)")
print("  = Gewichtete Spezifikation von 'was am wichtigsten ist'")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📏 Level 1: Exact Match
  Der einfachste Check: stimmt die Antwort exakt?
  'positive' == 'positive' → 1.0 ✅
  'positive' == 'negative' → 0.0 ❌

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📏 Level 2: Token F1 (Partial Credit)
  Was wenn die Antwort TEILWEISE stimmt?
  F1 balanciert Precision (wie viele Treffer waren korrekt?)
  und Recall (wie viele wurden gefunden?)

  🟡 Vorsichtig: nur 1 von 3 gefunden, aber der war richtig
     Gold: ['apple', 'banana', 'cherry'] | Pred: ['apple'] → F1 = 0.50
  🟡 Übereifrig: alles gefunden, aber 2 Falsche dabei
     Gold: ['apple', 'banana', 'cherry'] | Pred: ['apple', 'banana', 'cherry', 'grape', 'melon'] → F1 = 0.75
  ✅ Perfekt: genau richtig
     Gold: ['apple', 'banana', 'cherry'] | Pred: ['apple', 'banana', 'cherry'] → F1 = 1.00
  ❌ Komplett daneben: nichts stimmt
     Gold: ['apple', 'banana'] | Pred: ['cherry', 'grape'] → F1 = 0.00

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📏 L

## 4. Kann ich mit dem Prompt die Genauigkeit verbessern?

Du hast gesehen, dass das Modell Fehler macht. Und du weisst jetzt, wie man Genauigkeit misst. Die naheliegende Frage: **hilft eine bessere Formulierung?**

Probier's aus! Editiere den Prompt unten und klick "Auswerten". Jeder Versuch wird aufgezeichnet — so siehst du, ob deine Änderungen wirklich etwas bringen.


In [13]:
from dspy_tasks.visualize import prompt_workshop

# Vorausgefüllter Prompt — editiere ihn und sieh was passiert!
workshop = prompt_workshop(
    task_id="sentiment",
    default_instructions="Classify the sentiment of the product review as positive, negative, or neutral.",
    max_eval=10,
)
display(workshop)


### 💡 Und — hat es geholfen?

- Wurde dein Score besser?
- Wie viele Versuche hast du gebraucht?
- Hättest du das für 20 verschiedene Aufgaben machen wollen?

Genau das ist das Problem mit manuellem Prompt-Tuning: es **funktioniert** — aber es ist **langsam, mühsam und fragil**. Was bei einer Aufgabe hilft, verschlechtert vielleicht eine andere.

> Im nächsten Notebook sehen wir, wie man das **automatisiert**.


### Mit Benchmark-Daten testen

**TruthfulQA** ist ein Benchmark aus der KI-Forschung — Fragen, die LLMs besonders gerne falsch beantworten. Editiere den Prompt und versuch eine bessere Accuracy zu erreichen:


In [14]:
from dspy_tasks.benchmarks import load_truthfulqa, contains_match
from dspy_tasks.actions import run_on_examples
import dspy

truthful_examples = load_truthfulqa(8)

# Vorausgefüllter Prompt für TruthfulQA
tqa_prompt = widgets.Textarea(
    value="Answer the question accurately. Be careful about common misconceptions and myths. If the common belief is wrong, give the scientifically correct answer.",
    layout=widgets.Layout(width="100%", height="100px"))
tqa_label = widgets.HTML('<div style="font-weight:bold; margin-bottom:4px">✏️ Dein Prompt für Fakten-Fragen:</div>')
tqa_btn = widgets.Button(description="Auswerten!", button_style="primary", icon="play", layout=widgets.Layout(width="200px"))
tqa_out = widgets.Output()

class TQASig(dspy.Signature):
    """Placeholder"""
    question = dspy.InputField(desc="A factual question")
    answer = dspy.OutputField(desc="A truthful answer")

def on_tqa_run(b):
    with tqa_out:
        tqa_out.clear_output()
        result = run_on_examples(
            truthful_examples, tqa_prompt.value, TQASig, contains_match)
        display_score("TruthfulQA Score", result.score)
        display_results_table(result.individual_scores)

tqa_btn.on_click(on_tqa_run)
display(tqa_label, tqa_prompt, tqa_btn, tqa_out)

HTML(value='<div style="font-weight:bold; margin-bottom:4px">✏️ Dein Prompt für Fakten-Fragen:</div>')

Textarea(value='Answer the question accurately. Be careful about common misconceptions and myths. If the commo…

Button(button_style='primary', description='Auswerten!', icon='play', layout=Layout(width='200px'), style=Butt…

Output()

## 5. Kann das Modell Code schreiben?

Eine besonders spannende Aufgabe: das Modell soll **Python-Code generieren**. Hier wird Evaluation richtig interessant — denn wir können den generierten Code tatsächlich **ausführen** und prüfen ob er funktioniert!

Die Metrik prüft:
- Hat der Code eine Funktionsdefinition (`def`)?
- Gibt es ein `return`-Statement?
- Stimmen die Schlüsselwörter mit der erwarteten Lösung überein?


In [ ]:
task = get_task("code_generation")
print(f"📋 Task: {task.name}")
print(f"   Metrik: Strukturprüfung (def, return, Keywords)")
print()

# Lass uns ein paar Code-Aufgaben testen
examples = task.load_examples()[:5]
module = task.make_module()

for ex in examples:
    print(f"📝 Aufgabe: {str(ex.description)[:80]}")
    prediction = module(description=ex.description)
    code = prediction.python_code.strip()
    print(f"   Generierter Code:")
    for line in code.split('\n')[:6]:
        print(f"   │ {line}")
    if code.count('\n') > 5:
        print(f"   │ ... ({code.count(chr(10))+1} Zeilen)")
    
    # Versuche den Code auszuführen!
    try:
        exec(compile(code, '<generated>', 'exec'), {})
        print(f"   ✅ Code läuft ohne Fehler!")
    except Exception as e:
        print(f"   ❌ Fehler: {type(e).__name__}: {e}")
    
    score = task.metric_fn(ex, prediction)
    print(f"   📊 Metrik-Score: {score:.0%}")
    print()


## ⏭️ Weiter geht's!

Du hast Metriken, du kannst messen, du hast manuell getuned. Aber was, wenn der **Computer die Prompts SELBST optimieren** könnte?

👉 **[Weiter zu Notebook: Automatische Optimierung →](02_optimization.ipynb)**
